# Practice 3 TAB: Pacman with AlphaBeta + Neural Networks

### *Autores:* ***Sergio Azorín Julián y José Francisco Hurtado***

## Introducción

Esta práctica tiene como propósito fundamental diseñar e implementar un agente capaz de jugar a Pacman de una manera inteligente, integrando estrategias de anticipación al adverario (AlphaBeta) con modelos de aprendizaje automático (Red Neuronal).

Para la realización de la práctica hemos llevado a cabo un proceso evolutivo y acumulativo, en el que partimos de agentes que empleaban puramente la intuición de la red neuronal hacia sistemas híbridos que combinan el experiencia de las partidas previas con planificaciones estategicas que analizan la partida actual unos pasos por delante.

Iniciamos nuestro trabajo a partir de un agente neuronal puro (en nuestro caso NeuralAgentDummy), un agente entrenado a partir de un conjunto de partidas jugadas por nosotros. Esta forma de actuar se basaba únicamente en evaluar el tablero actual y seleccionar la acción con mayor probabilidad con respecto a su entrenamiento. Sin embargo, al no presentar ningún tipo de estrategia, este agente acaba siendo incapaz de prever trampas a corto, medio y largo plazo.

Con el fin de mitigar los fallos de supervivencia de nuestro modelo neuronal, se ha implementado un segundo agente en el que se complementa la salida de la red neuronal con un conjunto de heurísticas basadas, por ejemplo, en la distacia a la comida y a los fantasmas. A pesar de otorgar una mejora sobre el anterior agente, sigue manteniendo el mismo problema, no es capaz de anticiparse a los movimientos de los enemigos, teniendo un comportamiento puramente reactivo (greedy, horizonte de búsqueda de 0).

Para soluciónar esta cuestión se ha optado por emplear Minimax, un método de busqueda adversarial, el cual, a partir de un árbol que represente el conjunto de movimientos legales, podrémos encontrar la acción que más beneficia a Pacman suponiendo que los adversarios van a jugar óptimamente. Debemos de tener en cuenta que este tipo de busquedas presentan un coste computacional exponenecial a medida que aumentamos la profundidad de busqueda. Por ello, utilizaremos AlphaBeta como técnica de optimización, la cual reducirá el conjunto de nodos evaluados sin afectar al resultado final.

Con este nuevo tipo de técnicas somos capaces de superar el enfoque reactivo, otorgándole a nuestro agente inteligente la posibilidad de crear estrategias. Este agente definitivo nace con el proposito de fusionar la experiencia acumulada por la red neuronal y la anticipación a las jugadas del rival.

Para consolidar esta fusión de forma robusta, el agente implementa dos innovaciones clave:

- **Función de Evaluación Híbrida**: Que unifica en un solo valor las estimaciones de la red neuronal y las reglas heurísticas.

- **Estrategia de pesos dinámicos**: Una mejora que varía la importacia de cada componente según el progreso de la partida.

A través de este documento, se expondrá en mayor detalle la implementación de este agente definitivo (AlphaBetaNeuralAgent) y los resultados que hemos obtenido al otrogarle el control de la partida.

## **Parte 1. Nuevas Heurísticas**

### **1. Introducción**

#### Las heurísticas
Las heurísticas son como "intuiciones" matemáticas que tiene el agente para poder calcular / evaluar si la situación en la que se encuentra le favorece a él o al rival ¿cómo? pues teniendo una lista de directrices (buenas jugadas / costumbres) que le indican si la posición en cuestión le beneficia a él o no.

#### Aplicado al Pacman
En el caso del Pacman nos vamos a basar en factores relacionados con la obtención de la comida, evasión de los fantasmas y otras estrategias para evitar exponerse a situaciones vulnerables. Dependiendo de estos factores se irán sumando o restando puntos a una variable que será el score de la heurística siendo este el que indique como de favorable o desfavorable es la posición en cuestión para Pacman. Cuanto mayor sea el score mejor será la situación para él y viceversa, cuanto peor score peor situación.

### **2. Extracción de los datos de la partida**
Antes de empezar a aplicar las heurísticas necesitamos saber en que escenario se encuentra Pacman, por eso obtenemos todos los datos relevantes del estado de la partida. Es lo primero que se hace al llamar a nuestra función `traditional_evaluation`: 

#### Código

```Python
    score = state.getScore()
    pacman_pos = state.getPacmanPosition()
    food = state.getFood().asList()
    ghost_states = state.getGhostStates()
    capsules = state.getCapsules()
    legal_actions = state.getLegalActions()
```

#### ¿Qué es cada cosa?
- Score: Es la puntuación base que nos da el propio juego. La usamos como punto de partida. Sobre esta puntuación base, nosotros sumaremos o restaremos nuestros propios puntos de heurística para "guiar" a Pacman.

- pacman_pos: Es la coordenada exacta de nuestro Pacman (ej: x=5, y=3). Es nuestro punto de referencia para medir a qué distancia están las cosas buenas (comida) y las cosas malas (fantasmas).

- food: Aquí sacamos dónde están las bolitas de comida. Usamos .asList() para convertir el mapa del juego en una simple lista de coordenadas. Así es mucho más fácil y rápido calcular distancias luego.

- ghost_states: Este dato es clave, no solo obtenemos la posición de los fantasmas, sino su estado. En Pacman, un fantasma puede ser mortal (hay que huir) o puede estar asustado (hay que comérselo). Esta variable nos dirá si el temporizador de miedo (scaredTimer) está activo o no.

- capsules: Las coordenadas de las cápsulas.

- legal_actions: Nos dice qué movimientos puede hacer Pacman en este preciso lugar (Norte, Sur, Este, etc.)

#### ¿Cómo medimos las distancias entre objetos?
Para calcular cómo de lejos está Pacman de la comida o de los fantasmas, usamos la Distancia de Manhattan. Esta mide los pasos reales que hay que dar moviéndose en forma de cruz (arriba, abajo, izquierda, derecha).

### **3. Implementación de las heurísticas**
Hemos dividido nuestras heurísticas en tres categorías lógicas:

#### Categoría A: Navegación y recolección
##### Factor 1: Distancia a la comida más cercana

###### Código
```Python
    if food:
        min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
        score += 1.0 / (min_food_distance + 1)
```
Busca la bolita de comida que esté más cerca y le da a Pacman un pequeño incentivo para ir hacia ella. Usamos una fracción (1.0 dividido entre la distancia) para que, cuanto más cerca esté la comida, mayor sea la recompensa que reciba. Esto hace que priorice mucho más un coco que está a 1 movimiento de distancia que otro que está a 3.

Al tener un peso tan bajo conseguimos que quiera moverse hacia la comida pero que no se ciegue por ella y se exponga a situaciones de alto peligro tontamente.

##### Factor 4: Densidad local de comida
###### Código
```Python
    nearby_food = sum(1 for f in food if manhattanDistance(pacman_pos, f) <= 4)
    score += nearby_food * 2.0
```
En este caso estamos mirando en un radio de 4 posiciones alrededor del Pacman para ver si hay algún grupo de comida considerable cerca. De esta manera potenciamos que el Pacman quiera ir a zonas en las cuales tiene mucha comida a su alrededor. Es especialmente útil cuando está a la misma distancia de un coco en un sentido que en el otro pero en uno solo tiene ese coco y ya mientras que en el otro tiene muchos más cocos que podrá comerse. 

#### Categoría B: Supervivencia y Evasión
##### Factor 2: Proximidad directa a fantasmas
###### Código
```Python
    for ghost_state in ghost_states:
        ghost_pos = ghost_state.getPosition()
        ghost_distance = manhattanDistance(pacman_pos, ghost_pos)

        if ghost_state.scaredTimer > 0:
            # Si el fantasma está asustado, acercarse a él
            score += 50 / (ghost_distance + 1)
        else:
            # Si no está asustado, evitarlo
            if ghost_distance <= 2:
                score -= 200  # Gran penalización por estar demasiado cerca
```
Evalúa la amenaza de ser comido por los fantasmas o de comerse a los fantasmas. Si el fantasma está asustado, se le da una gran recompensa para que vaya a por él ya que este le va a dar un boost en la puntuación muy considerable. Sin embargo, si este es letal se penaliza fuertemente estar cerca de él.

##### Factor 5: Radar preventivo de fantasmas
###### Código
```Python
    for g in ghost_states:
        if g.scaredTimer == 0:
            dist = manhattanDistance(pacman_pos, g.getPosition())
            if 2 < dist <= 5:
                score -= 20.0 / dist
```
Este sistema actúa a modo de prevención / alerta temprana. A diferencia del factor 2 que tiene al fantasma muy cerca y le aplica una gran penalización este factor va a aplicar una penalización leve por estar próximo a un fantasma pero no encima. De esta manera el Pacman evitará acercarse sin más a zonas en las que están los fantasmas y darse cuenta de golpe que los tiene demasiado cerca para poder huir.

#### Categoría C: Táctica Avanzada y Control del Escenario (El Cerebro)
##### Factor 3: Cápsulas de poder (Modo Defensivo y Ofensivo)
```Python
    if capsules:
        min_cap_dist = min(manhattanDistance(pacman_pos, c) for c in capsules)
        # ¿Hay fantasmas peligrosos cerca?
        danger_close = any(
            g.scaredTimer == 0 and manhattanDistance(pacman_pos, g.getPosition()) <= 4
            for g in ghost_states
        )
        if danger_close:
            score += 100.0 / (min_cap_dist + 1)
        else:
            score += 5.0 / (min_cap_dist + 1)
```

Pacman evalúa el contexto antes de comerse una cápsula. Si no hay peligro cerca, la cápsula no le importa mucho (+5.0), prefiriendo guardarla para luego. Pero si detecta un fantasma letal a menos de 4 pasos, la cápsula se convierte en una vía de escape y su valor se dispara (+100.0), haciendo que Pacman quiera comerla para salvarse y de paso comerse a los fantasmas y ganar más puntos.

##### Factor 6: Anti-Caza Suicida
```Python
    for g in ghost_states:
        if g.scaredTimer > 0:
            dist = manhattanDistance(pacman_pos, g.getPosition())
            if dist > g.scaredTimer:
                score -= 50.0 / (dist + 1)

```
A veces, un fantasma está asustado, pero le quedan 2 segundos de miedo y nosotros estamos a 5 pasos de distancia. El Factor 2 nos haría perseguirlo, pero para cuando lleguemos el fantasma ya habrá despertado y nos encontraremos en una situación comprometida. Esta heurística compara la distancia hasta el fantasma asustado con el tiempo restante: si no vamos a llegar a tiempo, aplica un -50.0 que cancela instantáneamente las ganas de ir a cazarlo.

### **4. Balanceo de pesos**
Queremos destacar que simplemene tener buenas ideas para las heurísticas no sirve de nada si estas no están bien calibradas para trabajar conjuntamente. La razón por la que estas sí que funcionan en el Pacman reside en cómo interactúan los pesos de las 6 reglas entre sí cuando se suman. Hemos establecido una jerarquía de necesidades:

- La supervivencia importa más que comer: La comida normal da incentivos muy pequeños. Sin embargo, un fantasma cercano resta -200 puntos. Si Pacman ve una bolita de comida pero hay un fantasma letal a dos pasos, la suma de las heurísticas dará un número negativo grande. De esta manera, el agente entiende que comer ahí es un suicidio y decide huir. 

- Adaptación al contexto: El mejor ejemplo de nuestro balanceo es el Factor 3 (las cápsulas). En una situación sin fantasmas, la cápsula solo vale +5. Con este peso tan bajo, Pacman prefiere ir a limpiar zonas donde hay mucha comida normal agrupada (Factor 4). Sin embargo, si detecta peligro, el peso de la cápsula se multiplica y pasa a valer +100.

En resumen, los pesos garantizan que el Pacman actúe por este orden: 1º Sobrevivir (evitando valores muy negativos), 2º Aprovechar ventajas tácticas (cazar fantasmas asustados o usar cápsulas en emergencias por sus altos valores positivos) y 3º Recolectar comida (guiado por valores bajos cuando el entorno es seguro).


## **Parte 2. Entrenamiento de la Red Neuronal**

### Descripción del Dataset

Para que la red neuronal aprendiera a jugar a Pacman le hemos enseñado muchos ejemplos con diferentes partidas. Nuestro dataset está compuesto por 93 partidas completas, donde la inmensa mayoría han sido jugadas manualmente por un humano para asegurar un comportamiento lógico y estratégico, apoyadas por algunas partidas generadas automáticamente por nuestro agente híbrido AlphaBetaNeuralAgent.

El win-rate (tasa de victoria) de este dataset es del 100%. Esto es vital, ya que no queremos que la red aprenda de nuestros errores, garantizando así que imite únicamente estrategias que conducen a la victoria.

Además, aplicamos un balanceo y filtrado del dataset. Aunque todas las partidas son victorias, no todas son igual de buenas (ganar con 1700 puntos habiendo cazado fantasmas es mucho mejor que ganar con 1300 huyendo por el mapa). Por ello, aplicamos un filtro del Top 90% basado en la puntuación final obtenida en cada partida. Descartamos el 10% de las peores partidas para asegurarnos de que la red solo clona los comportamientos más eficientes y seguros, eliminando las partidas menos prometedoras o con menor potencial de aprendizaje.

### Preprocesamiento

La red neuronal no ve los gráficos del juego. En la función preprocess_maps, convertimos el tablero visual en una matriz numérica simple: las paredes son 0, los espacios vacíos 1, la comida 2, las cápsulas 3, etc.

Tras esto, aplicamos una normalización: dividimos todos esos números (por el valor máximo, 5.0) para que queden comprendidos estrictamente entre 0.0 y 1.0. A las redes neuronales les cuesta mucho trabajar con números grandes o escalas dispares; darles los valores normalizados hace que las matemáticas internas de la red fluyan mucho más rápido y el proceso de aprendizaje sea mucho más estable.


### La Arquitectura de la Red 

La red que estamos usando (la clase PacmanNet) es una Red Neuronal Prealimentada (Feedforward Neural Network) o Perceptrón Multicapa. Su estructura es muy lógica y consta de las siguientes fases:

1. La Capa de Entrada (Input): El tablero de Pacman es bidimensional (X, Y), pero las redes densas necesitan entradas unidimensionales. Por eso, lo primero que hace la red mediante la instrucción x.view(x.size(0), -1) es "aplanar" el tablero, convirtiendo la cuadrícula en una única y larga fila de números.

2. Las Capas Ocultas (El procesamiento): La red tiene dos capas internas (Hidden Layers).

    - La primera capa coge todos los datos aplanados del tablero y los conecta a 256 neuronas.

    - La segunda capa reduce y condensa esa información a 128 neuronas.

    - ¿Por qué se hace así? La primera capa busca patrones básicos visuales ("hay un fantasma cerca", "hay comida a la derecha"), y la segunda capa combina esos patrones en conceptos tácticos más abstractos ("estoy acorralado", "es buen momento para atacar").

3. Activación ReLU: Entre las capas, usamos la función ReLU, que nos permite introducir no-linealidad a la red neuronal para que pueda aprender patrones complejos. Básicamente, lo que hace es cambiar por 0 el valor de una neurona si este es negativo. Esto ayuda a la red a tomar decisiones claras y "apagar" las conexiones que no aportan información útil.

4. El Mecanismo de Dropout (Anti-memorización): En el código aplicamos un Dropout(0.3). Esto significa que, durante el entrenamiento, apagamos aleatoriamente el 30% de las neuronas en cada iteración. ¿Por qué haríamos algo así? Para evitar el temido Overfitting (sobreajuste). Si no hiciéramos esto, la red se aprendería el mapa de memoria en lugar de entender la lógica subyacente del juego. Al "cegar" temporalmente algunas neuronas, la forzamos a ser robusta y a saber jugar aunque la situación en el tablero sea nueva.

5. La Capa de Salida (Output): Finalmente, la información destilada llega a la última capa, que tiene exactamente 5 neuronas. Cada neurona representa una acción posible: Norte, Sur, Este, Oeste o Parar. La neurona que devuelva el valor más alto (tras aplicar una función Softmax para convertir los valores en probabilidades) será la acción que elija ejecutar el agente.

### Hiperparámetros de Entrenamiento

El entrenamiento es el proceso iterativo de ajustar las conexiones de la red neuronal para que acierte la salida deseada. En el archivo net.py hemos definido cómo va a aprender mediante estos parámetros:

- Epochs (200): Son las veces que la red va a repasar el dataset de entrenamiento al completo. Con 200 épocas, le damos tiempo suficiente para consolidar el conocimiento sin pasarnos.

- Batch Size (64): En lugar de ver las partidas de una en una, o todas de golpe, le pasamos "lotes" de 64 fotos del tablero a la vez. Es un equilibrio perfecto para no saturar la memoria gráfica/RAM del ordenador y mantener un ritmo de aprendizaje fluido.

- Learning Rate (0.001): Es el "tamaño del paso" que da la red al corregir sus errores. Un 0.001 es lo bastante pequeño para no saltarse la respuesta óptima por accidente, pero lo bastante grande para no tardar muchísimo tiempo en aprender.

- Split de Test (20%): De todos nuestros datos, apartamos un 20% que la red nunca ve durante la fase de estudio (test_size=0.2). Esto lo usamos como "examen final" objetivo para comprobar si la red realmente sabe jugar a Pacman o si solo ha memorizado las partidas de entrenamiento.

- Función de Pérdida (CrossEntropyLoss): Es la métrica que permite a la red evaluar sus propios errores, comparando la acción que predijo con la acción que realmente tomó el humano. En base a esa diferencia, y mediante el algoritmo de backpropagation, se modifican proporcionalmente los pesos de las capas de la red para que en la próxima iteración las predicciones se acerquen más a la decisión correcta. 

## Parte 3. AlphaBetaNeuralAgent

En esta sección se abordará el diseño y desarrolo de AlphaBetaNeuralAgent, el componente central de la práctica que integra la busqueda adversarial optimizada con AlphaBeta y la intuición adquirida por el modelo neuronal. Tal y como se ha expuesto anteriormente, este agente tiene como objetivo superar las limitaciones que presenta la toma de decisiones greedy, evaluando estados futuros a través de una función de puntuación que mezcla las heuriístcas con la predicción de la red neuronal.

In [ ]:
class AlphaBetaNeuralAgent(AlphaBetaAgent):
    
    def __init__(self,
                 evalFn='scoreEvaluationFunction',
                 depth='3',
                 start_heuristicsWeight=0.3,  # Peso de AlphaBeta al INICIO
                 start_nnWeight=0.7,          # Peso de la Red Neuronal al INICIO
                 end_heuristicsWeight=0.7,    # Peso de AlphaBeta al FINAL
                 end_nnWeight=0.3):           # Peso de la Red Neuronal al FINAL):
        super().__init__(evalFn, depth)

        self.evaluationFunction = self.evaluation_combined
        
        # Guardamos los límites de la transición
        self.start_w_heuristic = start_heuristicsWeight
        self.start_w_neural = start_nnWeight
        self.end_w_heuristic = end_heuristicsWeight
        self.end_w_neural = end_nnWeight
        
        # Pesos actuales que usará evaluation_combined
        self.w_heuristic = self.start_w_heuristic
        self.w_neural = self.start_w_neural
        
        self.neural_agent_dummy = NeuralAgentDummy()
        
        # Guardaremos la cantidad inicial de comida para calcular el progreso
        self.initial_food = None
    
    def evaluation_combined(self, state):
        # 1) Traditional score (with the new heuristics from Task 1)
        trad_score = traditional_evaluation(state)
        #print(f"Heuristic score: {trad_score}")

        # 2) Neural network score
        neural_score = self.neural_agent_dummy.neural_evaluation(state)
        #print(f"Neural network score: {neural_score}")

        # 3) Weighted combination
        return self.w_heuristic * trad_score + self.w_neural * neural_score


    def getAction(self, gameState: GameState):
        """
        Returns the minimax action using self.depth and self.evaluationFunction
        """
        "*** YOUR CODE HERE ***"
        # 1. Registrar la comida inicial en el primer turno
        if self.initial_food is None:
            self.initial_food = gameState.getNumFood()
            # Si el mapa empieza sin comida (raro, pero previene división por cero)
            if self.initial_food == 0:
                self.initial_food = 1 

        # 2. Calcular el progreso del juego (de 0.0 al inicio, a 1.0 al final)
        current_food = gameState.getNumFood()
        progress = 1.0 - (current_food / self.initial_food)

        # 3. Interpolación lineal de los pesos
        self.w_heuristic = self.start_w_heuristic * (1 - progress) + self.end_w_heuristic * progress
        self.w_neural = self.start_w_neural * (1 - progress) + self.end_w_neural * progress


        def alphabeta(agentIndex, depth, gameState, alpha, beta):

            if (gameState.isWin() or
                gameState.isLose() or
                depth == self.depth):
                return self.evaluationFunction(gameState)

            #Max (Pacman)
            if agentIndex == 0:
                return maxValue(agentIndex,depth,gameState,alpha,beta)

            #Min (Fantasmas)
            return minValue(agentIndex,depth,gameState,alpha,beta)

        def maxValue(agentIndex,depth,gameState,alpha,beta):

            v = float('-inf')
            legalActions = gameState.getLegalActions(agentIndex)

            if not legalActions:
                return self.evaluationFunction(gameState)

            for action in legalActions:
                successor = gameState.generateSuccessor(agentIndex,action)
                value = alphabeta(1,depth,successor,alpha,beta)
                v = max(v,value)

                #Hacemos poda
                if v >= beta:
                    return v

                alpha = max(alpha, v)

            return v


        def minValue(agentIndex,depth,gameState,alpha,beta):
            v = float('inf')


            legalActions = gameState.getLegalActions(agentIndex)

            if not legalActions:
                return self.evaluationFunction(gameState)

            nextAgent = agentIndex+1
            nextDepth = depth

            if nextAgent == gameState.getNumAgents():
                nextAgent = 0
                nextDepth = depth + 1

            for action in legalActions:

                successor = gameState.generateSuccessor(agentIndex,action)

                value= alphabeta(nextAgent,nextDepth,successor,alpha,beta)

                v = min(v,value)

                #Hacemos poda
                if v <= alpha:
                    return v

                beta = min(beta, v)

            return v

        alpha = float('-inf')
        beta = float('inf')

        bestAction = None
        bestScore = float('-inf')

        for action in gameState.getLegalActions(0):
            successor = gameState.generateSuccessor(0,action)

            score = alphabeta(1,0,successor,alpha,beta)

            if score > bestScore:
                bestScore = score
                bestAction = action

            alpha = max(alpha,bestScore)

        return bestAction

**Función de Evaluación Combinada**

La piedra angular de este agente híbrido es su mecanismo para determinar el valor de un estado de juego futuro. La función *evaluation_combined()* unifica dos componentes fundamentales:

- Puntuación Tradicional (trad_score): Evalúa el estado basandose en las heurísticas que hemos implementado y desarrollado en la parte 1 de este documento.

- Puntuación Neuronal (neural_score): Extrae el valor a partir de la experiancia asimilada durante el entrenamiento de la red neuronal.

Para determinar el valor definitivo, se unen las componentes mediante una suma ponderada definida de la siguiente manera:

$$final\_score = w_{heuristic} \cdot trad\_score + w_{neural} \cdot neural\_score$$

In [ ]:
def evaluation_combined(self, state):
         # 1) Traditional score (with the new heuristics from Task 1)
        trad_score = traditional_evaluation(state)

        # 2) Neural network score
        neural_score = self.neural_agent_dummy.neural_evaluation(state)

        # 3) Weighted combination
        return self.w_heuristic * trad_score + self.w_neural * neural_score

**Estrategia de Pesos Dinámicos**

Como mejora avanzada sobre los pesos estáticos y con el objetivo de maximizar el rendimiento , se ha diseñado una estrategia de pesos dinámicos dependientes del progreso de la partida. La implementación funciona de la siguiente manera: 

- Métrica de progreso: En el primer movimiento se registra la cantidad total de comida en la partida. En cada turno, se calcula un factor de progreso normalizado de $0.0$ a $1.0$ basado en la comida consumida (progress = 1.0 - (current_food / initial_food)).

- Transición de prioridades: Al comienzo del juego (progreso cercano a $0.0$), el agente da mayor peso a la intuición general de la red neuronal (start_nnWeight = 0.7 frente a start_heuristicsWeight = 0.3). A medida que el valor de *progress* aumenta, los pesos invierten sus magnitudes de forma gradual hasta ortorgar el control a las heurísticas (end_heuristicsWeight = 0.7 frente a end_nnWeight = 0.3)

- Justificación de la estrategia: Al inicio del juego la configuaración del mapa y las posiciones en las que se encuentra Pacman suelen ser muy similares entre partidas, es por ello que la red neuronal es capaz de ayudar a decidir cuales son los mejores movimientos para garantizar un buen cominezo, lo cual le permite a Pacman posicionarse en situaciones beneficiosas para él en un futuro no muy lejana, puediendo encontrar grandes clústeres de comida y acumular una gran puntuación lo antes posible. Sin embargo, en el endgame (cuando queda poca comida), las situaciones de tablero se vuelven inusuales respecto a la distribución general del dataset de entrenamiento, en otras palabras, es muy raro e improbable que la red haya juegado una partida que sea idéntica o muy similar a esa. En esta fase crítica priorizamos las heurísticas ya que aseguran que Pacman no cometa errores por falta de experiencia y recoja la comida restante de manera segura, aunque dedique un mayor tiempo.

In [ ]:
# 1. Registrar la comida inicial en el primer turno
if self.initial_food is None:
    self.initial_food = gameState.getNumFood()
    # Si el mapa empieza sin comida (raro, pero previene división por cero)
    if self.initial_food == 0:
        self.initial_food = 1 

# 2. Calcular el progreso del juego (de 0.0 al inicio, a 1.0 al final)
current_food = gameState.getNumFood()
progress = 1.0 - (current_food / self.initial_food)

# 3. Interpolación lineal de los pesos
self.w_heuristic = self.start_w_heuristic * (1 - progress) + self.end_w_heuristic * progress
self.w_neural = self.start_w_neural * (1 - progress) + self.end_w_neural * progress

**Integración del algoritmo Minimimax y poda AlphaBeta**

Para anticipar las interacciones futuras, el agente despliega un árbol de juego manjeado por la función interna *alphabeta*. Esta estructura busca maximizar la puntuación de Pacman, mientras que todos los fantasmas operan de forma coordinada para minimizarla.

La implementación refleja la dinámica de turnos y propagación de límites:

- **Gestión de Agentes y Profundidades**: El algoritmo alterna el control entre Pacman (agente 0) llamando a *maxValue*, y los fantasmas (agentes>0) llamando a *minValue*. La profundidad del árbol (depth) únicamente se incrementa cuando todos los fantasmas han actuado y el control retorna de nuevo a Pacman, definiendo así un turno completo.

In [ ]:
#Max (Pacman)
if agentIndex == 0:
    return maxValue(agentIndex,depth,gameState,alpha,beta)

#Min (Fantasmas)
return minValue(agentIndex,depth,gameState,alpha,beta)

- **Casos Base**: La recursión se detiene devolviendo de valor de evaluation_combined bajo tres condiciones estrictas: si Pacman gana el juego, si pierde la partida, o si se alcanza profundidad máxima (self.depth).

In [ ]:
if (gameState.isWin() or
    gameState.isLose() or
    depth == self.depth):
    return self.evaluationFunction(gameState)

- **Mecanismo de Poda**: El buen rendimiento computacional se garantiza mediante el mantenimiento de límites dinámicos $\alpha$ (la mejor alternativa explorada por Pacman) y $\beta$ (la mejor alternativa explorada por los fantasmas). 

En la función *maxValue*, si se descubre un estado futuro cuyo valor *v* es mayor o igual a $\beta$, se detiene la evaluación de los hermanos de ese nodo, ya que el jugador minimizador nunca permitirá que se alcance esa rama. 

De forma análoga, en la función *minValue*, si el valor desciende por debajo de $\alpha$, se poda el resto de la sub-rama.

In [ ]:
def maxValue(agentIndex,depth,gameState,alpha,beta):

    v = float('-inf')
    legalActions = gameState.getLegalActions(agentIndex)

    if not legalActions:
        return self.evaluationFunction(gameState)

    for action in legalActions:
        successor = gameState.generateSuccessor(agentIndex,action)
        value = alphabeta(1,depth,successor,alpha,beta)
        v = max(v,value)

        #Hacemos poda
        if v >= beta:
            return v

        alpha = max(alpha, v)

    return v


def minValue(agentIndex,depth,gameState,alpha,beta):
    v = float('inf')


    legalActions = gameState.getLegalActions(agentIndex)

    if not legalActions:
        return self.evaluationFunction(gameState)

    nextAgent = agentIndex+1
    nextDepth = depth

    if nextAgent == gameState.getNumAgents():
        nextAgent = 0
        nextDepth = depth + 1

    for action in legalActions:

        successor = gameState.generateSuccessor(agentIndex,action)

        value= alphabeta(nextAgent,nextDepth,successor,alpha,beta)

        v = min(v,value)

        #Hacemos poda
        if v <= alpha:
            return v

        beta = min(beta, v)

    return v

Finalmente, en el bucle principal o nodo raíz. Pacman itera sobre sus acciones legales, invoca la rutina *alphabeta* sobre los estados sucesores y selecciona aquella acción que retorna el valor definitivo (bestScore), garantizando así una decisión óptima generada tanto por cálculo de la heurísticas y la experiencia de la red neuronal.

In [ ]:
alpha = float('-inf')
beta = float('inf')

bestAction = None
bestScore = float('-inf')

for action in gameState.getLegalActions(0):
    successor = gameState.generateSuccessor(0,action)

    score = alphabeta(1,0,successor,alpha,beta)

    if score > bestScore:
        bestScore = score
        bestAction = action

    alpha = max(alpha,bestScore)

return bestAction

## Parte 4. Resultados

In [9]:
from texttable import Texttable
import statistics



old_layout_Greedy=[-436.0, -461.0, -477.0, -410.0, -85.0, -465.0, -478.0, -316.0, -276.0, -463.0]
new_layout_Greedy=[-459.0, -467.0, -455.0, -447.0, -192.0, -264.0, -463.0, -213.0, -402.0, -121.0]

old_layout_GreedyH=[-376.0, -368.0, 1650.0, 777.0, 533.0, 2031.0, 784.0, 167.0, 117.0, 1053.0]
new_layout_GreedyH=[-376.0, -368.0, 975.0, 412.0, 1510.0, 827.0, 440.0, -349.0, 606.0, 1644.0]

old_layout_AlphaBeta=[2053.0, 1872.0, 1899.0, 662.0, 1909.0, 1484.0, 2068.0, -351.0, 403.0, 945.0]
new_layout_AlphaBeta=[445.0, 1829.0, 1825.0, 1895.0, -367.0, 671.0, 1731.0, 1092.0, 874.0, 1727.0]


table = Texttable()
table.add_rows([["Configuration", "Classical layout", "New layout"],
               ["Greedy neural agent",f"score: {statistics.mean(old_layout_Greedy)} \nwin_rate: 0/10",f"score: {statistics.mean(new_layout_Greedy)} \nwin_rate: 0/10"],
               ["Greedy neural agent + heuristics",f"score: {statistics.mean(old_layout_GreedyH)} \nwin_rate: 2/10",f"score: {statistics.mean(new_layout_GreedyH)} \nwin_rate: 2/10"],
               ["AlphaBeta + NN + heuristics",f"score: {statistics.mean(old_layout_AlphaBeta)} \nwin_rate: 6/10",f"score: {statistics.mean(new_layout_AlphaBeta)} \nwin_rate: 5/10"]])
print(table.draw())

+----------------------------------+------------------+----------------+
|          Configuration           | Classical layout |   New layout   |
+==================================+==================+================+
| Greedy neural agent              | score: -386.7    | score: -348.3  |
|                                  | win_rate: 0/10   | win_rate: 0/10 |
+----------------------------------+------------------+----------------+
| Greedy neural agent + heuristics | score: 636.8     | score: 532.1   |
|                                  | win_rate: 2/10   | win_rate: 2/10 |
+----------------------------------+------------------+----------------+
| AlphaBeta + NN + heuristics      | score: 1294.4    | score: 1172.2  |
|                                  | win_rate: 6/10   | win_rate: 5/10 |
+----------------------------------+------------------+----------------+


A partir de la tabla comparativa generada tras la ejecución de al menos 10 partidas por configuración, se pueden extraer conclusiones muy claras sobre el impacto de cada componente introducido en la arquitectura del agente:  
- **El impacto de las nuevas heurísticas**: Tal y como sugiere la comparativa, añadir nuevas heurísticas mejoró el rendimiento de forma drástica. El modelo base de NeuralAgent (NeuralAgentDummy en nuestro código) presenta un desempeño deficiente, siendo incapaz de ganar una sola partida (tasa de victoria 0/10) y promediando puntuaciones fuertemente negativas (-386.7 en el mapa clásico), lo que indica muertes prematuras o bucles ineficientes. Al sumar las heurísticas personalizadas (NeuralAgent en nuestro código), la puntuación media se dispara a 636.8 puntos y se consiguen las primeras victorias (2/10). Esto demuestra que el modelo neuronal por sí solo carecía de conocimientos defensivos a corto plazo, una deficiencia que las heurísticas diseñadas logran mitigar. 

- **El impacto de la planificación (Alpha-Beta)**: La integración del algoritmo Minimax con Poda Alfa-Beta supuso el mayor salto cualitativo en el comportamiento del agente. Al abandonar la toma de decisiones pura del modo greedy para adoptar un razonamiento adversarial a múltiples pasos, la tasa de victorias se triplica en el entorno clásico (del 20% al 60%) y la puntuación promedio prácticamente se duplica, alcanzando los 1294.4 puntos. Esto confirma el objetivo principal de la práctica: la red neuronal se beneficia inmensamente de la capacidad de explorar un árbol de juego.  

- **Capacidad de generalización al nuevo mapa**: Es fundamental recordar que la red neuronal fue entrenada exclusivamente en la configuración mediumClassic. A pesar de que el nuevo mapa (customMaze) altera las dinámicas del juego al contar con áreas más abiertas donde antes había paredes , el agente híbrido generaliza extraordinariamente bien. En el agente final (AlphaBeta + NN + heuristics), la penalización por el cambio de entorno es mínima: la tasa de éxito solo desciende ligeramente (de 6/10 a 5/10 victorias) y la puntuación se mantiene en niveles competitivos (1172.2). Esto se debe a que la Poda Alfa-Beta y las reglas heurísticas son agnósticas al diseño específico del mapa, dotando al sistema de la flexibilidad necesaria para desenvolverse en entornos desconocidos para la red. 

Debemos de tener en cuenta que los resultados obtenidos en la tabla previamente mostrada, han sido conseguidos con un valor de profundidad igual a 3. Con el objetivo de comprobar la mejora que obtendríamos en el nuevo mapa si subiésemos el nivel de profundidad, hemos vuelto a ejecutar la configuración de AlphaBeta + NN + heuristics con depth = 4.

In [13]:
new_layout_AlphaBeta_depth4= [1930.0, 1437.0, 2088.0, 918.0, 1745.0, 1903.0, 886.0, 1864.0, 851.0, 2114.0]

table2 = Texttable()  
table2.add_rows([["Configuration","New layout"],
               ["AlphaBeta + NN + heuristics (depth=4)",f"score: {statistics.mean(new_layout_AlphaBeta_depth4)} \nwin_rate: 7/10"]])

    
print(table2.draw())

+---------------------------------------+----------------+
|             Configuration             |   New layout   |
+=======================================+================+
| AlphaBeta + NN + heuristics (depth=4) | score: 1573.6  |
|                                       | win_rate: 7/10 |
+---------------------------------------+----------------+


- **Incremento en la tasa de victorias**: El porcentaje de éxito ascendió del 50% (5/10) al 70% (7/10). Este aumento del 20% demuestra que anticipar un turno completo más (el movimiento de Pacman y la respuesta correspondiente de los fantasmas) permite al agente esquivar situaciones de encierro que a profundidad 3 resultaban invisibles o inevitables.

- **Mejora de la puntuación media**: La puntuación media en el nuevo mapa se disparó de 1172.2 a 1573.6 puntos (un incremento de más de 400 puntos). Esto se traduce en que Pacman no solo sobrevive más, sino que es capaz de limpiar el mapa de comida de manera mucho más eficiente y rápida, maximizando las bonificaciones de tiempo y maximizando el consumo de fantasmas asustados cuando se presenta la oportunidad.

Aunque aumentar la profundidad incrementa el número de nodos en el árbol de juego de forma exponencial, la Poda Alfa-Beta implementada demostró ser lo suficientemente eficiente como para absorber el coste computacional extra sin ralentizar drásticamente el tiempo de respuesta por turno del agente, mostrando una partida fluida y sin muchos "tirones".

## Conclusiones

De los experimentos y los resultados obtenidos, podemos sacar las siguientes conclusiones:

- **El trabajo en equipo entre intuición y anticipación**: Al principio, el agente que solo usaba la red neuronal jugaba de forma muy tonta: sabía ir a por la comida en general, pero caía en todas las trampas porque no miraba hacia el futuro ni tenía en consideración otros factores relativos a la partida como la distancia hacia los fantasmas. Al juntar la red neuronal con las heurísticas conseguimos mejorar el panorama ya que ahora el agente combina la experiencia con el contexto de la partida actual, sin embargo, sigue teníendo un gran inconveniente que es su falta de previsión, no analiza el juego 2 o 3 o más movimientos de antemano, simplemente juega el mejor movimiento ahora exponiendose a estar condenado en 4 movimientos por ejemplo. Finalmente si juntamos la red neuronal con las heurísticas y el algoritmo AlfaBeta conseguimos el equilibrio perfecto: la red neuronal le da al Pacman una buena intuición de hacia donde moverse en el mapa, las heurísticas le ayudan a evaluar mejor su entorno y actuar en consecuencia y el algoritmo AlfaBeta le ayuda a calcular los próximos pasos para ver cual será el mejor movimiento no ahora sino en X movimientos para evitar caer en trampas de las cuales no pueda escapar.

- **La ventaja de cambiar los pesos durante la partida**: La idea de cambiar los valores de los pesos en tiempo real ha supuesto una mejoría notable en el rendimiento del agente. Al principio de la partida, cuando el mapa es muy similar de partida a partida, dejamos que la red neuronal guíe al Pacman. Sin embargo, al final de la partida, cuando queda muy poca comida, el Pacman pasa a hacer caso a las fórumulas matemáticas y a las reglas que hemos programado. Esto evita que la red neuronal se equivoque en situaciones que simplemente desconoce al ser terreno inexplorado por ella, además asegura que Pacman se coma los últimos trozos de comida de manera segura.

- **Anticipar los movimientos funciona**: Cuando obligamos al algoritmo a mirar 4 pasos hacia adelante en lugar de 3, el cambio es realmente visible. Pacman gana más partidas (pasando del 50% de win_rate al 70%) y consigue más puntos. Esto pasa porque al mirar un turno más allá, Pacman puede ver las emboscadas que los fantasmas podrían hacerle antes de ocurran y es capaz de cambiar de dirección a tiempo. Además, gracias a la poda de AlfaBeta podemos evitar realizar cálculos extra y no congelar nuestro equipo por una carga computaciona excesivamente alta.

- **Funciona bien en mapas nuevos**: Otro aspecto positivo que podemos destacar de este agente es que ha sido capaz de adaptarse bastante bien a lo desconocido. Aunque la red neuronal se entrenó en un mapa especifico, el Pacman final fue capaz de jugar casi igual de bien en el mapa nuevo. Esto se debe a que, si la red neuronal se confunde por el cambio en la distribución de paredes, las reglas matemáticas y el algoritmo AlfaBeta actúan como un "seguro de vida" que lo mantiene a salvo. Además cabe destacar que el nuevo mapa es más permisivo con Pacman, es decir, se eliminan los callejones superior e inferior donde era muy frecuente que se quedara atrapado por los fantasmas y acabará muriendo sin poder hacer nada al respecto. En este nuevo layout al tener menos zonas peligrosas el agente puede recorrer el mapa con más seguridad y garantías de no ser acorralado.

## Comandos

Partida 1: AlphaBeta + NN + heuristics (depth=3) mapa original -> score: 2053

python pacman.py -p AlphaBetaNeuralAgent

Partida 2: AlphaBeta + NN + heuristics (depth=4) customLayout -> score: 1930

python pacman.py -l customMaze -p AlphaBetaNeuralAgent